In [3]:
%matplotlib widget
# Boilerplate import code for all libraries
# Changes to the precision require re-loading the kernel and need to be done before any op uses them.
import sphWarpCore_config as swc
from typing import Any
swc.configure(precision="float32", dim=Any) # precision: float16|half|float32|single|float64|double

import sphWarpCore as sph
from sphWarpCore.type_config import *
print(get_type_config()) # confirms active settings

# Initialize warp at this point
import warp as wp; wp.init()

import os
import torch
if torch.cuda.is_available(): # set the TORCH_CUDA_ARCH_LIST environment variable to the compute capability of the GPU for faster compiles
    os.environ['TORCH_CUDA_ARCH_LIST'] = f'{torch.cuda.get_device_properties(0).major}.{torch.cuda.get_device_properties(0).minor}'

import warnings
from tqdm import TqdmExperimentalWarning
warnings.filterwarnings("ignore", category=TqdmExperimentalWarning)
from tqdm.autonotebook import tqdm

# final import blocks that are generic
import matplotlib.pyplot as plt
from torch.profiler import profile, record_function, ProfilerActivity
import numpy as np
import math
import shlex    
import subprocess
import shutil

# custom SPH libraries
from integrators.integration import *
from sphWarpCore import *

# This library
from compressibleSPH import *

{'scalar_t': <class 'warp._src.types.float32'>, 'dim_t': typing.Any}
Warp 1.12.0 initialized:
   CUDA Toolkit 12.9, Driver 13.2
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA RTX PRO 500 Blackwell Generation Laptop GPU" (6 GiB, sm_120, mempool enabled)
   Kernel cache:
     /home/lu26029/.cache/warp/1.12.0


In [4]:
import math

nx = 200
dim = 2
L = 10
n_h = 4

xc = yc = 0
beta = 5
gamma = 1.4
P_infty = 1
rho_infty = 1
rho0 = 1

extraData = {
    'nx': nx,
    'dim': dim,
    'L': L,
    'n_h': n_h,

    'gamma': gamma,
    'rho0': rho0,

    'xc': xc,
    'yc': yc,
    'beta': beta,
    'P_infty': P_infty,
    'rho_infty': rho_infty
}

In [5]:
device = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
dtype = get_torch_precision()

config, integrator = buildConfig(
    domain = buildDomainDescription(L, dim, True, device, dtype),
    dim = dim,
    kernel = KernelFunctions.B7,
    targetNeighbors = n_h_to_nH(4, dim),
    supportMode = SupportScheme.KernelMeanSymmetric,
    gradientMode = GradientScheme.Difference,
    laplacianMode = LaplacianScheme.Brookshaw,
    integrationScheme = IntegrationSchemeType.rungeKutta2,
    samplingScheme = SamplingScheme.regular,
    device = device,
    dtype = dtype,
    dt = None,
    adaptiveDt = True,
    cflFactor=0.3,
)
config.nx = nx

config.minDt = 1e-8
# config.dx = L / (nx * 2)

scheme = CompressibleSPHScheme.CRKSPH
SimulationSystem, SimulationState, SimulationConfig, SimulationUpdate, fn, export_fn, import_fn = buildScheme(scheme)


schemeConfig = SimulationConfig()
schemeConfig.gamma = gamma
schemeConfig.rho0 = rho0


schemeConfig.viscositySwitchParams.scheme = ViscositySwitch.NoneSwitch
schemeConfig.adaptiveSupportScheme = AdaptiveSupportScheme.Owen
schemeConfig.adaptiveSupportCorrections = False


In [ ]:

# from warp import Kernel

# from compressibleSPH.config import SimulationConfig
# from compressibleSPH.utils import *
# from sphWarpCore import *
# import torch
# from compressibleSPH.system import *
# from compressibleSPH.modules import idealGasEOS, evaluateOptimalSupport
# from compressibleSPH.enumTypes import *

In [24]:
from compressibleSPH.modules.timestep.compressible import computeTimestep

def sampleYeeVortex(nr, buffer_rings, config, schemeConfig, extraData):
    xc = extraData['xc']
    yc = extraData['yc']
    beta = extraData['beta']
    gamma = extraData['gamma']
    P_infty = extraData['P_infty']
    rho_infty = extraData['rho_infty']
    rho0 = extraData['rho0']
    particles_, shells = sampleShellv2(nr, config.domain, config.targetNeighbors)

    # fig, axis = plt.subplots(1, 2, figsize=(11,5), squeeze=False)

    indices = torch.hstack([torch.ones(s[4].shape[0], device = particles_.positions.device, dtype = torch.int32) * s[0] for s in shells])
    # colors = torch.rand((len(shells), 3), device = particles_.positions.device)
    # sampledColors = colors[indices.long()]

    # r = torch.norm(particles_.positions, dim=1)

    # axis[0,0].scatter(particles_.positions[:,0].cpu(), particles_.positions[:,1].cpu(), s=5, c=sampledColors.cpu(), cmap = 'tab20')
    # axis[0,0].set_title('Particle Positions')

    r = torch.norm(particles_.positions, dim=1)

    deltav_term = beta / (2 * math.pi) * torch.exp( ( 1 - r**2 ) / 2)
    deltav_x = deltav_term * (- particles_.positions[:, 1] + yc)
    deltav_y = deltav_term * (particles_.positions[:, 0] - xc)

    deltaT = - ( ( gamma - 1) * beta**2 ) / ( 8 * gamma * math.pi**2 ) * torch.exp( 1 - r**2 )

    T_infty = P_infty / rho_infty

    T = T_infty + deltaT
    rhoInitial = T**(1 / (gamma - 1))
    Pinitial = rhoInitial * T
    uInitial = 1 / (gamma - 1) * (Pinitial / rhoInitial)


    actualMasses = rhoInitial / rho0 * particles_.masses

    v_initial = torch.stack([deltav_x, deltav_y], dim = 1)


    # from optimalSupport import evaluateOptimalSupport

    # dim = 2
    # domain = buildDomainDescription(L, dim, periodic = periodicDomain, device = device, dtype = dtype)

    # particles_ = sampleRegularParticles(nx, config.domain, config.targetNeighbors)

    # particles_ = sampleShell(32, config.domain, config.targetNeighbors, circle = True, extraRings=0)
    # print(f'Number of particles: {particles_.positions.shape[0]}')

    # domain = buildDomainDescription(domainExtent * 1.5, dim, periodic = periodicDomain, device = device, dtype = dtype)
    particles_ = particles_._replace(masses = particles_.masses * schemeConfig.rho0)
    L = config.domain.max[0] - config.domain.min[0]
    config.dx = L / nx


    particles = SimulationState(
        positions = particles_.positions,
        supports = particles_.supports,
        masses = actualMasses,
        densities = particles_.densities,
        velocities = v_initial,
        
        kinds = torch.zeros_like(particles_.positions[:,0], dtype = torch.int32),
        materials = torch.zeros_like(particles_.positions[:,0], dtype = torch.int32),
        UIDs = torch.arange(particles_.positions.shape[0], device = device, dtype = torch.int32),
        UIDcounter = particles_.positions.shape[0],
        
        internalEnergies = None,
        totalEnergies = None,
        entropies = None,
        pressures = None,
        soundspeeds = None,

        divergence=torch.zeros_like(particles_.densities),
        alpha0s= torch.ones_like(particles_.densities),
        alphas= torch.ones_like(particles_.densities),
    )


    densities = warpOperation(
        particles, 
        OperationProperties(
            kernel = config.kernel,
            operation = WarpOperation.Density,
            supportMode = SupportScheme.Gather,
            gradientMode = config.gradientMode,
            laplacianMode = config.laplacianMode,
        ),
        domain = config.domain,
    )
    particles.densities = densities

    compressibleSPHConfigAdapt = CompressibleSPHConfig(
        adaptiveSupportIterations=16,
        adaptiveSupportThreshold=1e-3,
        adaptiveSupportScheme=AdaptiveSupportScheme.NoScheme,
    )

    # neighborhood, neighbors = evaluateNeighborhood(particles, domain, wrappedKernel, verletScale = 1.0, mode = SupportScheme.SuperSymmetric, priorNeighborhood=None)
    # numNeighbors = coo_to_csr(filterNeighborhoodByKind(particles, neighbors.neighbors, which = 'noghost')).rowEntries
    # config = {'targetNeighbors': targetNeighbors, 'domain': domain, 'support': {'iterations': 16, 'scheme': 'Monaghan'}, 'neighborhood': {'algorithm': 'compact'}}
    # rho, h, rhos, hs, neighborhood = evaluateOptimalSupport(particles, wrappedKernel, neighborhood, SupportScheme.Gather, config)


    rho_optimal, h_optimal, adjacency, rhos_iter, supports_iter = evaluateOptimalSupport(particles, config, supportScheme = SupportScheme.Gather, compParams = compressibleSPHConfigAdapt)


    adjacency = buildVerletList(particles, domain = config.domain, verletScale = 1.0, supportMode = SupportScheme.SuperSymmetric, priorNeighborhood=None, verbose=False)

    print(f'Optimal Support min: {h_optimal.min()}, max: {h_optimal.max()}, mean: {h_optimal.mean()}')
    print(f'Number of neighbors min: {adjacency.numNeighbors.min()}, max: {adjacency.numNeighbors.max()}, mean: {adjacency.numNeighbors.float().mean()}')



    # compressibleSPHConfig = CompressibleSPHConfig(
    #     adaptiveSupportIterations=16,
    #     adaptiveSupportThreshold=1e-3,
    #     adaptiveSupportScheme=AdaptiveSupportScheme.Owen,
    # )

    # neighborhood, neighbors = evaluateNeighborhood(particles, domain, wrappedKernel, verletScale = 1.0, mode = SupportScheme.SuperSymmetric, priorNeighborhood=None)
    # numNeighbors = coo_to_csr(filterNeighborhoodByKind(particles, neighbors.neighbors, which = 'noghost')).rowEntries
    # config = {'targetNeighbors': targetNeighbors, 'domain': domain, 'support': {'iterations': 16, 'scheme': 'Monaghan'}, 'neighborhood': {'algorithm': 'compact'}}
    # rho, h, rhos, hs, neighborhood = evaluateOptimalSupport(particles, wrappedKernel, neighborhood, SupportScheme.Gather, config)

    rho_optimal, h_optimal, adjacency, rhos_iter, supports_iter = evaluateOptimalSupport(particles, config, supportScheme = SupportScheme.Gather, compParams = compressibleSPHConfigAdapt)



    adjacency = buildVerletList(particles, domain = config.domain, verletScale = 1.0, supportMode = SupportScheme.SuperSymmetric, priorNeighborhood=None, verbose=False)

    print(f'Optimal Support min: {h_optimal.min()}, max: {h_optimal.max()}, mean: {h_optimal.mean()}')
    print(f'Number of neighbors min: {adjacency.numNeighbors.min()}, max: {adjacency.numNeighbors.max()}, mean: {adjacency.numNeighbors.float().mean()}')



    # particleState.supports = h_optimal

    particles.densities = rho_optimal
    particles.supports = h_optimal

    # P_initial = torch.ones_like(particles.densities)
    u = 1 / (gamma - 1) * (Pinitial / rho_optimal)
    # A_, u_, P_, c_s = idealGasEOS(A = None, u = None, P = P_initial, rho = rho_optimal, gamma = gamma)
    A_, u_, P_, c_s = idealGasEOS(A = None, u = u, P = None, rho = rho_optimal, gamma = gamma)

    internalEnergy = u_ 
    kineticEnergy = torch.linalg.norm(torch.zeros_like(particles.positions), dim = -1) **2/ 2
    totalEnergy = (internalEnergy + kineticEnergy) * particles.masses

    simulationState_ = SimulationState(
        positions = particles.positions,
        supports = particles.supports,
        masses = particles.masses,
        densities = particles.densities,        
        velocities = v_initial,

        kinds = torch.zeros_like(particles_.positions[:,0], dtype = torch.int32),
        materials = torch.zeros_like(particles_.positions[:,0], dtype = torch.int32),
        UIDs = torch.arange(particles_.positions.shape[0], device = device, dtype = torch.int32),
        UIDcounter = particles.positions.shape[0],
        
        internalEnergies = u_,
        totalEnergies = totalEnergy,
        entropies = A_,
        pressures = P_,
        soundspeeds = c_s,

        alphas = torch.ones_like(particles.densities),
        alpha0s = torch.ones_like(particles.densities),
        divergence=torch.zeros_like(particles.densities),
    )

    # area = L**dim / (nx**dim)
    # rho_low = 1
    # rho_high = 2

    # mask = torch.logical_and(particles.positions[:,0].abs() < L/4, particles.positions[:,1].abs() < L/4)

    # simulationState_.masses[mask] = area * rho_high
    # simulationState_.densities[mask] = rho_high
    # simulationState_.masses[~mask] = area * rho_low
    # simulationState_.densities[~mask] = rho_low

    rho_optimal, h_optimal, adjacency, rhos_iter, supports_iter = evaluateOptimalSupport(particles, config, supportScheme = SupportScheme.Gather, compParams = compressibleSPHConfigAdapt)
    # simulationState_.densities = rho_optimal
    adjacency = buildVerletList(
        simulationState_, 
        config.domain, verletScale = 1.4, supportMode = SupportScheme.SuperSymmetric,
        priorNeighborhood = None,
        verbose = False)

    apparentVolume, simulationState_.densities, crkState = computeCRKFactors(simulationState_, config.domain, config.kernel, adjacency = adjacency)


    # P_initial = torch.ones_like(particles.densities)
    # u = 1 / (gamma - 1) * (P_initial / simulationState_.densities)
    # A_, u_, P_, c_s = idealGasEOS(A = None, u = None, P = P_initial, rho = rho_optimal, gamma = gamma)
    A_, u_, P_, c_s = idealGasEOS(A = None, u = None, P = Pinitial, rho = simulationState_.densities, gamma = gamma)
    simulationState_.internalEnergies = u_
    simulationState_.pressures = P_
    simulationState_.soundspeeds = c_s


    print(f"min density: {simulationState_.densities.min()}, max density: {simulationState_.densities.max()}")
    print(f"min mass: {simulationState_.masses.min()}, max mass: {simulationState_.masses.max()}")
    print(f"min support: {simulationState_.supports.min()}, max support: {simulationState_.supports.max()}")

    adjacency = buildVerletList(simulationState_, 
                            domain = config.domain,
                            verletScale = 2**(1/config.dim), supportMode = config.supportMode)

    compressibleSystem = SimulationSystem(
        state=simulationState_, 
        adjacency = adjacency, 
        domain = config.domain)


    config.dt = computeTimestep(compressibleSystem, config, schemeConfig, dt = None)
    initialState = (v_initial, rhoInitial, uInitial)

    def YeeVelocity(t_, x, initialState):
        return initialState[0]
    def YeeDensity(t_, x, initialState):
        return initialState[1]
    def YeeInternalEnergy(t_, x, initialState):
        return initialState[2]
    def YeeAcceleration(t_, x, initialState):
        return torch.zeros_like(x)

    mask = indices >= (len(shells) - buffer_rings)
    print(f'Buffer Particles: {mask.sum()} out of {len(particles_.positions)} [ {mask.sum() / len(particles_.positions) * 100:.2f}% ]')

    def buffer_sdf(position):
        dist = torch.ones_like(position[:,0])
        dist[mask] = -1
        # dist[:] = -1
        return dist
    def buffer_sdf_gradient(position):
        # for the gradient we set the non buffer particles to point outwards, and the buffer particles to point inwards
        dist = position.clone()
        dist[mask] = -dist[mask]
        return torch.nn.functional.normalize(dist, dim=1)

    sdf = buffer_sdf(particles_.positions)
    sdf_gradient = buffer_sdf_gradient(particles_.positions)

    yeeBC = BoundaryCondition(
        type = BoundaryConditionType.dynamic,
        sdf = lambda x: (buffer_sdf(x), buffer_sdf_gradient(x)),
        dirichletFunctions = {
            'velocities': lambda state, cfg, schemeCfg, positions, d, n, t, dt: YeeVelocity(t, positions, initialState),
            'densities': lambda state, cfg, schemeCfg, positions, d, n, t, dt: YeeDensity(t, positions, initialState),
            'internalEnergies': lambda state, cfg, schemeCfg, positions, d, n, t, dt: YeeInternalEnergy(t, positions, initialState),
        },
        updateFunctions = {
            'dvdt': lambda state, cfg, schemeCfg, positions, d, n, t, dt: YeeAcceleration(t, positions, initialState),
            'dxdt': lambda state, cfg, schemeCfg, positions, d, n, t, dt: YeeVelocity(t, positions, initialState),
        }
    )


    return compressibleSystem, indices, yeeBC
        

In [25]:
compressibleSystem, indices, yeeBC = sampleYeeVortex(32, 10, config, schemeConfig, extraData)

schemeConfig.boundaryConditions.clear()
schemeConfig.boundaryConditions.append(yeeBC)

enforceDirichlet(compressibleSystem, compressibleSystem.t, config.dt, config, schemeConfig)

Optimal Support min: 0.6153044104576111, max: 0.639579176902771, mean: 0.6167129278182983
Number of neighbors min: 27, max: 56, mean: 47.463077545166016
Optimal Support min: 0.6153044104576111, max: 0.639579176902771, mean: 0.6167129278182983
Number of neighbors min: 27, max: 56, mean: 47.463077545166016
min density: 0.5303765535354614, max density: 1.0001040697097778
min mass: 0.012687571346759796, max mass: 0.023858949542045593
min support: 0.6153044104576111, max support: 0.639579176902771
Buffer Particles: 1745 out of 3304 [ 52.81% ]


In [27]:
runningState = compressibleSystem.initializeNewState()

kineticEnergy = 0.5 * (torch.linalg.norm(runningState.state.velocities, dim = -1) **2 * runningState.state.masses).sum()
thermalEnergy = (runningState.state.internalEnergies * runningState.state.masses).sum()
totalEnergy = kineticEnergy + thermalEnergy

In [ ]:
caseName = '10-Yee_Vortex'
exportPath = prepExport(f'{caseName}', config, schemeConfig, scheme, export_fn)
exportSimulationSystem(exportPath, 'initialState', scheme, compressibleSystem, exportAdjacency = False, stages = None, exportStagesAdjacency = False, extraData = dict({
    'kineticEnergy': kineticEnergy,
    'thermalEnergy': thermalEnergy,
    'totalEnergy': totalEnergy,
    'frame_num': 0,
}, **extraData))


In [30]:
from warpPlot import visualize, PlottingOptions, PlotScaling, GridVisualization, UniformColorMap, Mapping, DivergingColorMap
markerSize = 32
plotter = visualize(
    particleState = runningState.state,
    domain = config.domain,
    quantities = {
        "A": runningState.state.velocities,
        "B": runningState.state.densities,
    },
    plotOptions = {
        "A": PlottingOptions(
            colorMap = UniformColorMap.viridis,
            markerSize = markerSize,
            midPoint = 0.0,
            quantityScaling = PlotScaling.Linear,
            mapping = Mapping.L2Norm,
            plotTitle = "velocities",
            # gridVisualization = GridVisualization(
            #     resolution = 512,
            # ),
            # vMin=1e-10
        ),
        "B": PlottingOptions(
            colorMap = UniformColorMap.cividis,
            flipColorMap=True,
            markerSize = markerSize,
            midPoint = 0.0,
            quantityScaling = PlotScaling.Linear,
            plotTitle = "densities",
            # gridVisualization = GridVisualization(
            #     resolution = 512,
            # ),
        ),
    },
    figTitle = "Wave Equation Example",
    mosaic = 'AB',
    figsize= (11,5),
    backend='vispy',
    # backend='pyVista',
    # backendOptions = {
    #     # In notebooks, use trame for reliable live updates.
    #     'jupyter_backend': 'trame',
    # }
)

# if args.exportImages:
#     plotter.export(f'output/{folderName}/frame_00000.png', dpi = args.figureDpi)
imagePath = f'{exportPath}/images'
os.makedirs(imagePath, exist_ok = True)
plotter.export(f'{imagePath}/frame_00000.png', dpi = 300)

RFBOutputContext()

In [31]:
# config.dt = 2.5e-3
t_limit = 8.0
nSteps = int(t_limit / config.dt)

print(f"Running with dt: {config.dt}, which gives nSteps: {nSteps}")
# nSteps = 256

runningState = compressibleSystem.initializeNewState()

trajectory = []

priorStep = None
for i in (tq := tqdm(range(nSteps), leave = True)):
    begin = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    begin.record()
    result = integrator.function(
        state = runningState,
        f = fn,
        dt = config.dt,  
        config = config,
        compParams = schemeConfig,
        verbose = False,
        # priorStep = priorStep
    )
    end.record()
    torch.cuda.synchronize()
    priorStep = result.stages[-1]
    timing = begin.elapsed_time(end)

    runningState = result.state
    kineticEnergy = 0.5 * (torch.linalg.norm(runningState.state.velocities, dim = -1) **2 * runningState.state.masses).sum()
    thermalEnergy = (runningState.state.internalEnergies * runningState.state.masses).sum()
    totalEnergy = kineticEnergy + thermalEnergy

    trajectory.append(
        (i, (i+1)*config.dt, totalEnergy.item(), kineticEnergy.item(), thermalEnergy.item(), timing)
,     )


    if i % 10 == 0 and i > 0:
        plotter.updateQuantities(
            {
                "A": runningState.state.velocities,
                "B": runningState.state.densities,
            },
            newParticleState = runningState.state,
        )
        plotter.export(f'{imagePath}/frame_{i:05d}.png', dpi = 300)
        
    if i % 500 == 0:
        exportSimulationSystem(exportPath, f'state_{i:04d}', scheme, runningState, exportAdjacency = False, stages = result.stages, exportStagesAdjacency = True, extraData = dict(**extraData, **{
            'kineticEnergy': kineticEnergy,
            'thermalEnergy': thermalEnergy,
            'totalEnergy': totalEnergy,
            'frame_num': i,
        }))

        
    maxVel = torch.linalg.norm(runningState.state.velocities, dim = -1).max()
    tq.set_description(f"Step {i+1}/{nSteps}, time: {(i+1)*config.dt:8.4g}/{t_limit:8.4g}, TE: {totalEnergy:.3g}, KE: {kineticEnergy:.3g}, IE: {thermalEnergy:.3g} | max vel: {maxVel:.3g} | iter time: {timing:.3f} ms")
    # t = {runningState.t:2f}, dt = {config.dt:.3g}, ptcls = {len(runningState.state.positions)}\nTotal Energy: {totalEnergy:.3g}, Kinetic Energy: {kineticEnergy:.3g}, Thermal Energy: {thermalEnergy:.3g}'
    # break

Running with dt: 0.009999999776482582, which gives nSteps: 800


  0%|          | 0/800 [00:00<?, ?it/s]

Module compressibleSPH.modules.adaptiveSupport.wp_psi 8875fe6 load on device 'cuda:0' took 6.59 ms  (cached)
Module compressibleSPH.modules.adaptiveSupport.wp_psi0 37e5819 load on device 'cuda:0' took 7.76 ms  (cached)
Module sphWarpCore.operations.wp_density f0357bf load on device 'cuda:0' took 5.60 ms  (cached)
Module sphWarpCore.operations.wp_gradient a50c471 load on device 'cuda:0' took 8.98 ms  (cached)


/home/lu26029/dev/compressibleSPH/src/compressibleSPH/modules/adaptiveSupport/owenLUT.py:7: UserWarning: torch.searchsorted(): boundary tensor is non-contiguous, this will lower the performance due to extra data copy when converting non-contiguous tensor to contiguous, please use contiguous boundary tensor if possible. This message will only appear once per program. (Triggered internally at /pytorch/aten/src/ATen/native/BucketizationUtils.h:38.)
  ileft = (torch.searchsorted(xvalues, x, right = True) - 1).clamp(min = 0, max = len(xvalues) - 2)


Module compressibleSPH.modules.crk.accel 9056830 load on device 'cuda:0' took 8.38 ms  (cached)
Module compressibleSPH.modules.crk.dudt 7e9ded3 load on device 'cuda:0' took 11.55 ms  (cached)
Module compressibleSPH.modules.compSPH.balance 8be6a68 load on device 'cuda:0' took 5.99 ms  (cached)


In [32]:
exportSimulationSystem(exportPath, f'finalState', scheme, runningState, exportAdjacency = False, stages = result.stages, exportStagesAdjacency = True, extraData = dict(**extraData, **{
    'kineticEnergy': kineticEnergy,
    'thermalEnergy': thermalEnergy,
    'totalEnergy': totalEnergy,
    'frame_num': i,
}))

In [33]:
ffmpeg_cmd = "ffmpeg -y -loglevel error -hide_banner -framerate 50 -f image2 -pattern_type glob -i 'frame_*.png' -c:v libx264 -pix_fmt yuv420p -b:v 10M output.mp4"
subprocess.run(shlex.split(ffmpeg_cmd), check=True, cwd = imagePath)
ffmpeg_cmd = 'ffmpeg -y -loglevel error -hide_banner -i output.mp4  -vf "fps=50,scale=540:-1:flags=lanczos,palettegen" palette.png'
subprocess.run(shlex.split(ffmpeg_cmd), check=True, cwd = imagePath)
ffmpeg_cmd = 'ffmpeg -y -loglevel error -hide_banner -i output.mp4 -i palette.png -filter_complex "fps=25,scale=540:-1:flags=lanczos[x];[x][1:v]paletteuse" out.gif'
subprocess.run(shlex.split(ffmpeg_cmd), check=True, cwd = imagePath)

# now copy the output.mp4 and out.gif to the parent directory for easier access
shutil.copy(f'{imagePath}/output.mp4', f'{exportPath}/output.mp4')
shutil.copy(f'{imagePath}/out.gif', f'{exportPath}/out.gif');